# Objetivo

Desenvolvimento do Teleutils em ambiente de teste local

# Configuração do Ambiente

## Bibliotecas

In [ ]:
# 1. Módulos da Biblioteca Padrão
import logging
from pathlib import Path

# 2. Módulos de Terceiros
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import pandas_udf
from pyspark.sql import types as T # ArrayType, StructType, StructField, StringType, BooleanType

# 3. Teletools/Teleutils
from teleutils.core.extractors import CDRParquetExtractor, CDRTextExtractor
from teleutils.core.transformers import CDRTransformer

In [ ]:
# Configuração do Pandas
# Exibir todas as colunas
pd.set_option("display.max_columns", None)

# Exibir todas as linhas
pd.set_option("display.max_rows", None)

# Evitar que o texto nas células seja cortado
pd.set_option("display.max_colwidth", None)

In [ ]:
# Configuração para exibir logs no output da célula
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
)

# Reduzir ruído de bibliotecas externas (pyspark, py4j, etc.)
logging.getLogger("py4j").setLevel(logging.WARNING)
logging.getLogger("pyspark").setLevel(logging.WARNING)

## Spark

In [ ]:
spark = (
    SparkSession
        .builder
        .appName("SessaoLocal")
        .master("local[12]")
        .config("spark.driver.memory", "24g")
        .getOrCreate()
)
spark

## Arquivos de dados

### Dados de origem

In [ ]:
folder_tim = Path("/assets/data/cdr/teleparser/Semana96/tim/fixa/ngn_italtel")
files_tim = [str(file) for file in folder_tim.glob("*.parquet")]

### Arquivos intermediário e final

In [ ]:
extracted_file = "/assets/data/cdr/teletools/db/extraidos/tim_ngn_huawei_extraido.parquet"
transformed_file = "/assets/data/cdr/teletools/db/transformados/tim_ngn_huawei_transformado.parquet"

# Análise

## Dados brutos

In [ ]:
df_raw = spark.read.parquet(*files_tim[:2])
df_raw.limit(5).toPandas()

In [ ]:
columns_to_keep = [
    "network_call_reference_sequence_number", # referencia
    "call_id_number_time", # data_hora_referencia
    "starting_date_of_call", # _data
    "starting_time_of_call", # _hora
    "duration_of_call_seconds", # duracao
    "calling_information_calling_number", # numero_origem
    "called_information_called_number", # numero_destino
    "call_final_status_information_final_status", # resultado_chamada
    "trunking_group_identity_information_incoming_trunk_group_identity", # rota_entrada
    "trunking_group_identity_information_outgoing_trunk_group_identity", # rota_saida
    "network_call_reference_switch_identity", # bilhetador
    "type_of_call", # tipo_chamada
]
df = df_raw.select(columns_to_keep)
df.limit(5).toPandas()

## Extração

In [ ]:
%%time
extractor = CDRParquetExtractor(spark)
extractor.extract(files_tim, extracted_file, "stfc_italtel_ngn_tim")

In [ ]:
df_extracted = spark.read.parquet(extracted_file)
df_extracted.show(5,truncate=False)

## Transformação

In [ ]:
transformer = CDRTransformer(spark)
transformer.transform_stfc_italtel_ngn_tim(extracted_file, transformed_file)

In [ ]:
df_transformed = spark.read.parquet(transformed_file)
df_transformed.limit(5).toPandas().dropna(how="all",axis=1)

In [ ]:
df_transformed.groupBy("no_resultado_chamada").count().show(truncate=False)